## Original SCB agent prompt provided in the paper
aligning on what has been done in the SCB paper

In [1]:
from prompts.scb import get_original_scb_prompt

# Pass the current checkpoint number to be implemented 
print(get_original_scb_prompt(1, "circopt"))


Implement a program that 100% solves the specification.
That is all you need to do.

Use a virtual environment and ensure that a 'requirements.txt' is present with any dependencies
you need to solve the problem.

You are working on the following issue:
Issue path: checkpoint_1.md
Issue implementation path: checkpoint_1/

The entrypoint file must be named "circopt.py".



## Reader agent

In [2]:
from prompts.reader import get_reader_prompt

print(get_reader_prompt(2))


You are a senior software engineer that analyses modules in a software project.

Your job is to understand the following directory: 
Project root: agent_workspace
Directory: checkpoint_1/
Dependency graph: checkpoint_1_graph.json

Identify the existing modules in the codebase, using the dependency graph as reference. For each module, identify its responsibility. Do not propose new modules.

Output a JSON object that describes each module, using the following schema. If there are no modules, return an empty JSON array.

{
  "type": "array",
  "items": {
    "type": "object",
    "properties": {
      "module_name": {
        "description": "The name of the module.",
        "type": "string"
      },
      "module_path": {
        "description": "The directory path of the module.",
        "type": "string"
      },
      "responsibility": {
        "description": "A sentence describing the single responsibility of this module.",
        "type": "string"
      }
    },
    "required": [


In [3]:
READER_OUTPUT = [
    {
      "module_name": "launch",
      "module_path": "checkpoint_1/launch.py",
      "responsibility": "Serves as the top-level entry point script that bootstraps and invokes the pipeline CLI."
    },
    {
      "module_name": "pipeline",
      "module_path": "checkpoint_1/pipeline/",
      "responsibility": "Acts as the package initializer that exposes the pipeline processing system as an importable module."
    },
    {
      "module_name": "pipeline.ast_nodes",
      "module_path": "checkpoint_1/pipeline/ast_nodes.py",
      "responsibility": "Defines all AST node dataclasses used to represent parsed pipeline task definitions and expression trees."
    },
    {
      "module_name": "pipeline.lexer",
      "module_path": "checkpoint_1/pipeline/lexer.py",
      "responsibility": "Tokenizes raw .pipe source text into a stream of typed tokens for use by the parser."
    },
    {
      "module_name": "pipeline.parser",
      "module_path": "checkpoint_1/pipeline/parser.py",
      "responsibility": "Parses a token stream produced by the lexer into a dictionary of TaskDef AST nodes representing the full pipeline definition."
    },
    {
      "module_name": "pipeline.expr_parser",
      "module_path": "checkpoint_1/pipeline/expr_parser.py",
      "responsibility": "Parses token lists from success and requires blocks into expression and statement ASTs."
    },
    {
      "module_name": "pipeline.evaluator",
      "module_path": "checkpoint_1/pipeline/evaluator.py",
      "responsibility": "Evaluates expression and statement ASTs against runtime context such as params, workspace, environment variables, and job output."
    },
    {
      "module_name": "pipeline.executor",
      "module_path": "checkpoint_1/pipeline/executor.py",
      "responsibility": "Orchestrates end-to-end task execution by running shell scripts, resolving requires dependencies, evaluating success criteria, and recording job results."
    },
    {
      "module_name": "pipeline.main",
      "module_path": "checkpoint_1/pipeline/main.py",
      "responsibility": "Provides the CLI entry point that parses arguments and config, loads the pipeline file, and drives the executor."
    }
  ]

## Analyzer agent

## IMPORTANT! TEST THIS AGAIN USING THE EXTRACTED / PROCESSED METRICS NOT THE PATH


In [4]:
from prompts.analyzer import get_analyzer_prompt

# The actual DPy results should be filtered for specific things we concern only. 
# For this TEST example, a path is provided instead. DO NOT do this in your final submission 
print(get_analyzer_prompt(2, READER_OUTPUT, None, "checkpoint_1_dpy_metrics", None))


You are a senior software code quality analyst. 

Your job is to analyse the following directory: 
Project root: agent_workspace
Directory: checkpoint_1/

You are given a list of existing or new modules below. Some includes suggested improvements. 
[{'module_name': 'launch', 'module_path': 'checkpoint_1/launch.py', 'responsibility': 'Serves as the top-level entry point script that bootstraps and invokes the pipeline CLI.'}, {'module_name': 'pipeline', 'module_path': 'checkpoint_1/pipeline/', 'responsibility': 'Acts as the package initializer that exposes the pipeline processing system as an importable module.'}, {'module_name': 'pipeline.ast_nodes', 'module_path': 'checkpoint_1/pipeline/ast_nodes.py', 'responsibility': 'Defines all AST node dataclasses used to represent parsed pipeline task definitions and expression trees.'}, {'module_name': 'pipeline.lexer', 'module_path': 'checkpoint_1/pipeline/lexer.py', 'responsibility': 'Tokenizes raw .pipe source text into a stream of typed tok

In [ ]:
ANALYZER_OUTPUT = {
  "result": "fail",
  "improvements": [
    {
      "module_name": "pipeline.lexer",
      "smell": "The `tokenize` method is excessively long (139 lines) with a cyclomatic complexity of 38, containing a complex conditional with 4 conditions for number parsing.",
      "improvement_instruction": "Decompose `tokenize` into focused private helper methods, one per token category: `_scan_comment`, `_scan_dollar_expr`, `_scan_string`, `_scan_number`, `_scan_identifier`, `_scan_operator`. Each helper should handle its own token category and append to the token list, reducing `tokenize` to a dispatch loop."
    },
    {
      "module_name": "pipeline.parser",
      "smell": "`parse_task` is too long (87 lines, CC=23) and `parse_success_block` has high complexity (CC=10); additionally, `parse_pipeline_file` is flagged for feature envy as it also orchestrates lexer logic.",
      "improvement_instruction": "Split `parse_task` by extracting one private method per field handler: `_parse_run_field`, `_parse_success_field`, `_parse_requires_field`, `_parse_output_field`, `_parse_timeout_field`. Split `parse_success_block` into `_parse_success_inline_expr` and `_parse_success_braced_block`. Move the file-reading and lexer-invocation logic out of `parse_pipeline_file` so it only coordinates the `Parser` call, keeping lexing concerns inside `lexer`."
    },
    {
      "module_name": "pipeline.expr_parser",
      "smell": "`ExprParser` is insufficiently modularized (NOPM=22, WMC=85); `parse_for` has CC=17 with four complex conditionals; `parse_stmt` and `parse_primary` are both too long (67 and 84 lines respectively).",
      "improvement_instruction": "Break `parse_for` into three private helpers: `_parse_for_init`, `_parse_for_condition`, `_parse_for_update`, reducing its body to a coordination sequence. Split `parse_stmt` by extracting `_parse_decl_stmt` (for typed variable declarations) and `_parse_assign_stmt` (for assignments and increment/decrement). Split `parse_primary` by extracting `_parse_primary_literal`, `_parse_primary_dollar`, and `_parse_primary_ident` to handle each primary expression group."
    },
    {
      "module_name": "pipeline.evaluator",
      "smell": "`_exec_stmt` has a cyclomatic complexity of 26; multiple methods (`eval_block`, `_exec_stmt`, `_eval_func`, `_resolve_io_source`) contain empty catch blocks that silently swallow exceptions.",
      "improvement_instruction": "Replace all bare `except` or `except Exception: pass` blocks with explicit exception types and either re-raise, log, or return a meaningful sentinel value. Decompose `_exec_stmt` by extracting `_exec_if_stmt`, `_exec_for_stmt`, `_exec_while_stmt`, and `_exec_assign_stmt` as private methods, so `_exec_stmt` becomes a dispatch table with a single `isinstance` check per branch."
    },
    {
      "module_name": "pipeline.executor",
      "smell": "`Executor` has multiple responsibilities (multifaceted abstraction, LCOM=1): it handles shell execution, requires evaluation, success criterion evaluation, cycle detection, and job recording; `_exec_requires_stmt` has CC=18; `_evaluate_success_criterion` is flagged for feature envy (belongs in `evaluator`); `Executor` directly accesses private `Evaluator` members (`_eval`, `_exec_stmt`, `_truthy`); `JobContext.__init__` has 8 parameters; magic numbers `124` and `2` are used inline.",
      "improvement_instruction": "Move `_evaluate_success_criterion` into the `Evaluator` class and expose it as a public method, eliminating the cross-class private-member access. Replace the magic number `124` with a named constant `TIMEOUT_EXIT_CODE = 124` and `2` with a named constant `ESCAPE_STEP = 2`. Reduce `JobContext.__init__` to at most 5 parameters by grouping `workspace`, `cwd`, and `output_dir` into a `PathContext` dataclass. Decompose `_exec_requires_stmt` by extracting `_exec_requires_if`, `_exec_requires_for`, and `_exec_requires_while` helpers. Consider separating shell execution logic (`_execute_run`, `_build_shell_script`, `_process_string_escapes`) into a dedicated `ShellRunner` class to address the multifaceted abstraction smell."
    },
    {
      "module_name": "pipeline.main",
      "smell": "`main` is too long (70 lines, CC=15) and `_split_args` has CC=12 with a magic number `2`; two empty catch blocks silently discard `ValueError` in `_parse_literal`.",
      "improvement_instruction": "Extract config-loading logic from `main` into a `_load_config(path)` helper that returns an env dict and entry invocation string. Replace the two silent `except ValueError: pass` blocks in `_parse_literal` with explicit fall-through logic (try int, then float, then return the raw string) without exception swallowing. Replace the magic number `2` in `_split_args` with a named constant `ESCAPE_ADVANCE = 2`."
    },
    {
      "module_name": "pipeline",
      "smell": "Feature concentration: the package mixes five independent concern clusters (lexer/parser, evaluator, ast_nodes, executor/main, expr_parser) in a single flat package, yielding an LCC of 0.71.",
      "improvement_instruction": "Reorganise the package into sub-packages that align with the independent clusters identified: `pipeline/frontend/` for `lexer`, `parser`, and `ast_nodes`; `pipeline/runtime/` for `evaluator`, `expr_parser`, and `executor`; keep `pipeline/main.py` as the sole CLI entry point. Update all intra-package imports accordingly."
    }
  ]
}

## Decomposer Agent